# 5. Handling reduction sums (`gemv`)

`gemv` computes y = A·x for N × K matrix A and K vector x. Each output element is a sum,
and when its terms are spread over many banks, the partial sums have to be brought
together. The mapping decides how:

- **packing**: several partial sums share one row (`pack_factor`), folded into
  one by copies and adds inside the bank (`fold_levels`);
- **the tree across banks** (`reduce_tree`): at each level, some banks copy their
  partial result to another and it is added there, halving the banks involved.

A copy between two banks of one channel is a single command (`ROWCOPY`). A copy
between channels is a row read out of one controller and written into the other.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # tutorial/, for _infra
from _infra import pud

NAME, BANKS, MEM = "gemv_2048_2048", 64, "ddr"
k = pud.compile_kernel(NAME, BANKS, MEM, trace=True)
per_channel = pud.MEMORIES[MEM][1]
for loop in pud.mapping_attrs(k.mapping.path):
    tree = loop.pop("reduce_tree", None)
    print({key: v for key, v in loop.items()})
    if tree:
        print("\nreduce tree: copies per level", [len(level) for level in tree])
        for depth, level in enumerate(tree):
            cross = sum(s // per_channel != d // per_channel for s, d in level)
            print(f"  level {depth}: e.g. bank {level[0][0]:3} -> {level[0][1]:3}"
                  f"   ({cross} of {len(level)} copies cross a channel)")

Six levels fold 64 banks into one: 32 copies, then 16, 8, 4, 2 and 1. Only the
last level crosses from the second channel into the first; everything before it
stays inside a channel.

In the generated program, a level is not written out copy by copy: copies that
repeat across channels become one loop over channels and banks. That keeps the
program small enough for the instruction cache.

In [ ]:
text = k.pud.read_text()
print("copy instructions in the program:", text.count("pud.row_copy"),
      " cross-channel halves:", text.count("pud.row_stream"))

## Run it

In [ ]:
c = pud.simulate(k.exe, MEM)
t = pud.replay(k.mapping.trace, MEM)
print(f"compiled: {c.row_ops} commands, {c.makespan_us:9.3f} us  (simulated in {c.seconds:.0f} s)")
print(f"trace:    {t.row_ops} commands, {t.makespan_us:9.3f} us")
print("counts", "match" if c.row_ops == t.row_ops else "DIFFER")

> **Try it:** `BANKS = 32` keeps the whole tree inside one channel. Compare the
> ratio of compiled to trace with the 64-bank case, and with notebook 04.

## Where this stands

Every program in this tutorial issues exactly the commands its cost model estimated,
and gem5 now carries out what those commands do to the rows. What is still missing
is getting a kernel's **data** in and out of the bit-sliced layout, so that a
compiled kernel's results can be checked against a reference. 